# 03 — Champion Model Results

Requires `uv run python -m src.models.train` to have completed first.

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import shap
from sklearn.metrics import (
    roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay, confusion_matrix,
    roc_auc_score, average_precision_score,
)

from src.models.champion import load_champion
from src.models.train import load_splits
from src.explain.shap_utils import (
    compute_shap, mean_abs_shap, save_shap_arrays,
    REASON_CODES, feature_to_reason_code,
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

model = load_champion()
X_train, y_train, X_val, y_val, feature_cols = load_splits(Path('../data/processed'))
y_score = model.predict_proba(X_val)[:, 1]

metrics = json.loads(Path('../docs/results/champion_metrics.json').read_text())
print(f"ROC-AUC : {metrics['val_metrics']['roc_auc']}")
print(f"PR-AUC  : {metrics['val_metrics']['pr_auc']}")
print(f"FDR @3% : {metrics['val_metrics']['fdr_at_3pct_review']:.2%}")
print(f"FDR @5% : {metrics['val_metrics']['fdr_at_5pct_review']:.2%}")

## 1. ROC Curve & PR Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_val, y_score)
prec, rec, _ = precision_recall_curve(y_val, y_score)
roc_auc = roc_auc_score(y_val, y_score)
pr_auc = average_precision_score(y_val, y_score)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, color='#4C8EDA', lw=2, label=f'Champion XGBoost (AUC = {roc_auc:.4f})')
axes[0].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend()

baseline = y_val.mean()
axes[1].plot(rec, prec, color='#E05C5C', lw=2, label=f'Champion XGBoost (PR-AUC = {pr_auc:.4f})')
axes[1].axhline(baseline, color='k', linestyle='--', lw=1, label=f'No-skill baseline ({baseline:.3%})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

plt.suptitle('Champion Model — Validation Set Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/results/roc_pr_curves.png', bbox_inches='tight', dpi=150)
plt.show()

## 2. Threshold Selection & Confusion Matrix

Threshold set at the 3% review rate — the assumed operational review capacity.

In [ ]:
n_review_3pct = int(len(y_score) * 0.03)
threshold_3pct = float(np.sort(y_score)[::-1][n_review_3pct - 1])
y_pred = (y_score >= threshold_3pct).astype(int)

cm = confusion_matrix(y_val, y_pred)
tp, fp, fn, tn = cm[1,1], cm[0,1], cm[1,0], cm[0,0]

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=['Legitimate','Fraud']
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix @ 3% Review Rate (threshold = {threshold_3pct:.4f})')
plt.tight_layout()
plt.savefig('../docs/results/confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Threshold        : {threshold_3pct:.4f}')
print(f'Transactions reviewed : {n_review_3pct:,}  ({n_review_3pct/len(y_val):.1%})')
print(f'True Positives   : {tp:,}  ({tp/(tp+fn):.1%} recall)')
print(f'False Positives  : {fp:,}  ({fp/(fp+tp):.1%} false discovery rate)')
print(f'Precision        : {tp/(tp+fp):.2%}')

## 3. FDR vs Review Rate

In [ ]:
df_tbl = pd.DataFrame(metrics['threshold_table'])

fig, ax = plt.subplots(figsize=(10, 4))
ax2 = ax.twinx()
ax.plot(df_tbl['review_rate_pct'], df_tbl['fdr']*100,  'o-', color='#E05C5C', lw=2, label='FDR')
ax2.plot(df_tbl['review_rate_pct'], df_tbl['precision']*100, 's--', color='#4C8EDA', lw=2, label='Precision')
ax.set_xlabel('Review Rate (% of transactions)'); ax.set_ylabel('Fraud Detection Rate (%)', color='#E05C5C')
ax2.set_ylabel('Precision (%)', color='#4C8EDA')
ax.set_title('FDR and Precision vs Review Rate')
lines = ax.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax.legend(lines, labels)
plt.tight_layout()
plt.savefig('../docs/results/fdr_vs_review_rate.png', bbox_inches='tight', dpi=150)
plt.show()

print(df_tbl[['review_rate_pct','threshold','fraud_caught','fdr','precision','n_reviewed']].to_string(index=False))

## 4. SHAP Summary Plot

Computed on a 3,000-row stratified sample of the validation set.

In [ ]:
shap_values, X_sample = compute_shap(model, X_val, sample_n=3000, random_state=42)
save_shap_arrays(shap_values, X_sample, output_dir=Path('../data/processed'))
print(f'SHAP computed — {len(X_sample):,} rows × {shap_values.shape[1]} features')

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, max_display=20, show=False, plot_size=(10,8))
plt.title('SHAP Summary — Top 20 Features (n=3,000 val sample)', pad=12)
plt.tight_layout()
plt.savefig('../docs/results/shap_summary.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → docs/results/shap_summary.png')

## 5. SHAP Dependence Plots — Top 3 Features

In [ ]:
importance_df = mean_abs_shap(shap_values, feature_cols)
top3 = importance_df['feature'].head(3).tolist()
print('Top 3 by mean |SHAP|:', top3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, top3):
    shap.dependence_plot(feat, shap_values, X_sample, ax=ax, show=False, alpha=0.3, dot_size=8)
    ax.set_title(f'{feat}', fontsize=10)

plt.suptitle('SHAP Dependence — Top 3 Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/results/shap_dependence_top3.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Reason Code Coverage

In [ ]:
top20 = importance_df.head(20).copy()
top20['reason_code'] = top20['feature'].map(feature_to_reason_code)
top20['reason_text'] = top20['reason_code'].map(REASON_CODES)
print(top20[['feature','mean_abs_shap','reason_code','reason_text']].to_string(index=False))